# LeWM + RATISS — couplage environnemental PushT

Ce notebook ne remplace pas LeWM. Il conserve sa prédiction vanilla et ajoute le sidecar RATISS : loi de transition PushT, `P_sig` du moteur topologique RATISS, cohésion, thermodynamique et cache invariant.

> Le bras couplé optimise `MSE_LeWM + gate_RATISS × thermo_proxy`. `P_sig` reste un signal non différentiable de contrôle et de diagnostic ; il ne remplace jamais la loss LeWM.

In [ ]:
!pip -q install torch torchvision transformers omegaconf hydra-core einops pandas pyarrow opencv-python-headless ripser imageio stable-worldmodel stable-pretraining
!git clone --depth 1 https://github.com/lucas-maes/le-wm /content/le-wm
!git clone --depth 1 https://github.com/samajonathan9-source/ratiss-lewm-integration /content/ratiss-lewm-integration
!git clone --depth 1 https://github.com/samajonathan9-source/ratiss-topological-decoherence-engine /content/ratiss-topological-decoherence-engine

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import sys, subprocess, json, csv, time, random
import numpy as np, pandas as pd, torch
DRIVE=Path('/content/drive/MyDrive/ratiss_lewm_coupled'); DATA=DRIVE/'data'; CK=DRIVE/'checkpoints'; LOG=DRIVE/'logs'
for p in (DATA,CK,LOG): p.mkdir(parents=True,exist_ok=True)
SEED=123; EPOCHS=30; BATCH=8; LR=1e-4; WINDOWS_PER_EPOCH=256; LOG_EVERY=10; SAVE_EVERY=5; THERMO_WEIGHT=0.05
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'; print(DEVICE)

In [ ]:
def get(url,out):
    if not Path(out).exists(): subprocess.run(['wget','-q','-c',url,'-O',str(out)],check=True)
get('https://huggingface.co/datasets/lerobot/pusht/resolve/main/data/chunk-000/file-000.parquet',DATA/'data.parquet')
get('https://huggingface.co/datasets/lerobot/pusht/resolve/main/videos/observation.image/chunk-000/file-000.mp4',DATA/'pusht.mp4')
get('https://huggingface.co/quentinll/lewm-pusht/resolve/main/config.json',DATA/'config.json')
get('https://huggingface.co/quentinll/lewm-pusht/resolve/main/weights.pt',DATA/'weights.pt')
df=pd.read_parquet(DATA/'data.parquet'); state2=np.stack(df['observation.state'].to_numpy()).astype('float32'); action2=np.stack(df['action'].to_numpy()).astype('float32'); ep=df.episode_index.to_numpy(); n=len(df)
windows=[]
for e in range(int(ep.max())+1):
    ids=np.flatnonzero(ep==e); starts=np.unique(np.linspace(0,max(0,len(ids)-4),min(5,max(1,len(ids)-3)),dtype=int))
    for s in starts:
        q=ids[s:s+4]
        if len(q)==4: windows.append(q)
windows=np.asarray(windows,dtype='int64'); raw=subprocess.check_output(['ffmpeg','-v','error','-i',str(DATA/'pusht.mp4'),'-f','rawvideo','-pix_fmt','rgb24','-'])
frames=np.frombuffer(raw,dtype='uint8').reshape(n,96,96,3); X=np.asarray([frames[q] for q in windows],dtype='uint8')
A=np.zeros((len(windows),4,10),dtype='float32'); A[:,:,:2]=action2[windows]; S=state2[windows]
print('windows',len(windows),'frames',len(frames),'X',X.shape)

In [ ]:
sys.path[:0]=['/content/le-wm','/content/ratiss-lewm-integration','/content/ratiss-topological-decoherence-engine/src']
from omegaconf import OmegaConf
from hydra.utils import instantiate
from transformers import ViTConfig,ViTModel
from ratiss_lewm.pusht_coupling import RATISSPushTCoupler,PushTEnvironmentLaw,PushTLawConfig
def build():
    m=instantiate(OmegaConf.load(DATA/'config.json'))
    m.encoder=ViTModel(ViTConfig(hidden_size=192,num_hidden_layers=12,num_attention_heads=3,intermediate_size=768,image_size=224,patch_size=14,num_channels=3),add_pooling_layer=False)
    sd=torch.load(DATA/'weights.pt',map_location='cpu',weights_only=False); km={'.encoder.layer.':'.layers.','.attention.attention.query.':'.attention.q_proj.','.attention.attention.key.':'.attention.k_proj.','.attention.attention.value.':'.attention.v_proj.','.attention.output.dense.':'.attention.o_proj.','.intermediate.dense.':'.mlp.fc1.','.output.dense.':'.mlp.fc2.'}; norm={}
    for k,v in sd.items():
        for a,b in km.items(): k=k.replace(a,b)
        norm[k]=v
    m.load_state_dict(norm,strict=True); return m.to(DEVICE)
def tensors(ids):
    z=torch.from_numpy(X[ids]).permute(0,1,4,2,3).float().to(DEVICE)/255.0; a=torch.from_numpy(A[ids]).to(DEVICE); return z,a
law=PushTEnvironmentLaw(PushTLawConfig(psig_threshold=0.12)); print(law.name)

In [ ]:
def coupled_metrics(coupler,enc,pred,ids):
    out=[]
    for i,g in enumerate(ids):
        # transition law uses the real observed state/action and next observed state
        j=int(windows[g,0]); k=int(windows[g,1])
        out.append(coupler.observe({'emb':enc['emb'][i:i+1]},pred[i:i+1],state2[j],action2[j],state2[k],float(df.iloc[k]['next.reward']),bool(df.iloc[k]['next.done']),{'index':int(g),'episode':int(ep[j])}))
    return out
def run_coupled():
    torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED); model=build(); opt=torch.optim.AdamW(model.parameters(),lr=LR); coupler=RATISSPushTCoupler(law)
    path=CK/'coupled_latest.pt'; log=LOG/'coupled.csv'; rows=[]; start=0
    if path.exists():
        c=torch.load(path,map_location=DEVICE); model.load_state_dict(c['model']); opt.load_state_dict(c['opt']); rows=c.get('rows',[]); start=c['epoch']+1
    if not log.exists():
        with open(log,'w',newline='') as f: csv.DictWriter(f,fieldnames=['epoch','step','mse','coupled_loss','psig','physical_consistency','thermo_drift','cohesion','cache_rate']).writeheader()
    for epoch in range(start,EPOCHS):
        ids=np.random.default_rng(SEED+epoch).choice(len(X),size=min(WINDOWS_PER_EPOCH,len(X)),replace=False); t=time.time()
        for step in range(0,len(ids),BATCH):
            b=ids[step:step+BATCH]; z,a=tensors(b); enc=model.encode({'pixels':z,'action':a}); pred=model.predict(enc['emb'][:,:3],enc['act_emb'][:,:3]); target=enc['emb'][:,1:4]
            mse=torch.mean((pred-target)**2); metrics=coupled_metrics(coupler,enc,pred,b); gate=float(np.mean([x['coupling_gate'] for x in metrics])); drift=float(np.mean([x['thermodynamics']['prediction_drift'] for x in metrics])); psig=float(np.mean([x['topology']['p_sig'] for x in metrics])); phys=float(np.mean([x['transition']['physical_consistency'] for x in metrics])); coh=float(np.mean([x['cohesion']['entropy'] for x in metrics])); cache=float(np.mean([x['cache_accepted'] for x in metrics]))
            coupled_loss=mse + gate*THERMO_WEIGHT*torch.mean((pred-pred.mean(dim=-1,keepdim=True))**2); opt.zero_grad(); coupled_loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
            if step%(LOG_EVERY*BATCH)==0:
                row={'epoch':epoch,'step':step,'mse':float(mse.detach()),'coupled_loss':float(coupled_loss.detach()),'psig':psig,'physical_consistency':phys,'thermo_drift':drift,'cohesion':coh,'cache_rate':cache}; rows.append(row)
                with open(log,'a',newline='') as f: csv.DictWriter(f,fieldnames=row.keys()).writerow(row)
        if epoch%SAVE_EVERY==0 or epoch==EPOCHS-1: torch.save({'epoch':epoch,'model':model.state_dict(),'opt':opt.state_dict(),'rows':rows},path)
        print('coupled epoch',epoch,'mse',row['mse'],'psig',psig,'physical',phys,'saved',path)
    return pd.DataFrame(rows)
coupled_log=run_coupled()

In [ ]:
r=pd.read_csv(LOG/'coupled.csv'); summary=pd.DataFrame({'metric':['last_mse','best_mse','mean_psig','mean_physical_consistency','mean_thermo_drift','mean_cohesion','cache_rate'],'value':[r.mse.iloc[-1],r.mse.min(),r.psig.mean(),r.physical_consistency.mean(),r.thermo_drift.mean(),r.cohesion.mean(),r.cache_rate.mean()]}); summary.to_csv(LOG/'coupled_summary.csv',index=False); display(summary)
print('Couplage terminé. Les fichiers sont dans',LOG)